Para cargar el cuaderno en Google Colab en la esquina superior izquierda -> Archivo -> Subir cuaderno

# 🧰 HERRAMIENTAS DE WEB SCRAPING

---

## 📦 Librería `requests`
- Permite hacer **peticiones HTTP** desde Python.
- Es la forma de “pedirle una página web” a un servidor.

---

#### 🌐 Página de ejemplo
Vamos a usar una web diseñada para practicar web scraping, donde hay muchas citas listas para extraer.
https://quotes.toscrape.com

---

In [ ]:
import requests

r = requests.get("https://quotes.toscrape.com")

print(r.status_code)
print(r.text[:5000])

---
#### 📡 `r.status_code`
Nos indica el **estado de la petición HTTP**.

##### 📊 Códigos más comunes:

| Código | Significado |
|--------|-------------|
| 200 | Todo OK ✔️ |
| 404 | Página no encontrada ❌ |
| 403 | Acceso prohibido 🚫 |
| 429 | Demasiadas peticiones ⏳ |
| 500 | Error interno del servidor 💥 |

---

#### 📄 `r.text`
- Contiene la **respuesta completa del servidor en formato texto**
- Es decir, **todo el HTML de la página**
---
#### 🧠 Idea clave

> `requests` no extrae datos, solo descarga el HTML  
> El scraping real empieza después, al analizar ese HTML


---

## 📦 Librería `BeautifulSoup`
- Permite analizar y recorrer **HTML de forma sencilla**
- Convierte el HTML en una estructura navegable (DOM)
- Es la herramienta principal para extraer información en scraping

> "`requests` descarga la web, `BeautifulSoup` la interpreta"

---

#### ⚙️ ¿Qué es lo que hace?

- Convierte el HTML en un **árbol de elementos (DOM)**
- Permite recorrer la página como una estructura organizada
- Facilita la extracción de información específica

---

#### 🌳 HTML y estructura de la web

- El contenido de una página web está escrito en **HTML**
- HTML organiza la información mediante **etiquetas**
- BeautifulSoup nos permite acceder a esas etiquetas

Ejemplos de etiquetas:
- `<p>` → párrafos  
- `<h1>`, `<h2>` → títulos  
- `<div>`, `<span>` → contenedores  

---

#### 🔎 Ejemplo en HTML
Lo primero que tenemos que hacer es encontrar la etiqueta que queremos buscar.
En este caso:
```html
<span class="text">“Ejemplo de cita”</span>
 ```  
---
#### 💡 Idea clave

*   BeautifulSoup no “lee” el HTML como texto plano.
*   Lo convierte en una estructura organizada (DOM) para poder **navegar y extraer información de forma precisa dentro de la página**.



In [ ]:
from bs4 import BeautifulSoup
import requests

#Peticion
html = requests.get("https://quotes.toscrape.com").text

#Parser
soup = BeautifulSoup(html, "html.parser")

#Scrapping
quotes = soup.find_all("span", class_="text")

#Mostramos los 3 primeros
for q in quotes[:3]:
    print(q.text)


## 📦 Librería `Playwright`
- Permite controlar un **navegador real desde Python**
- Se utiliza para **web scraping dinámico**
- Simula el comportamiento de un usuario real

👉 A diferencia de `requests`, la página se renderiza completamente como en Chrome

---

#### 🧠 ¿Qué hace diferente a Playwright?

- Ejecuta JavaScript como un navegador real  
- Permite ver contenido dinámico  
- Puede interactuar con la página (clicks, scroll, etc.)  
- Es más potente que `requests + BeautifulSoup` en webs modernas  

---

#### 🌐 Página de ejemplo

https://books.toscrape.com

Vamos a extraer:
- 📖 Títulos de libros  
- 💰 Precio de cada libro  

---
#### ⚡ Programación asíncrona (`async / await`)

- Permite ejecutar tareas sin bloquear el programa  
- Mejora el rendimiento en automatización  

---

#### 🌍 ¿Qué es Chromium?

- Navegador que usa Playwright internamente  
- Renderiza la web como Chrome  
- Ejecuta JavaScript  
- Permite ver la página como un usuario real  

---

#### 🔁 Flujo de scraping con Playwright

- Se abre el navegador (Chromium)
- Se carga la página objetivo
- Se espera a que el contenido termine de cargar
- Se localizan los elementos (libros)
- Se extraen título y precio
- Se cierra el navegador

---

#### 🧠 Idea clave

Scraping moderno basado en **automatización de navegador real**

👉 Ideal para:
- Webs con JavaScript
- Bots avanzados
- Testing automatizado
- Scraping profesional

In [ ]:
!pip install playwright
!playwright install chromium
!playwright install-deps chromium

In [ ]:
import asyncio  # Permite usar programación asíncrona (async/await)
from playwright.async_api import async_playwright  # API asíncrona de Playwright

# Definimos una función asíncrona (obligatorio en Playwright async)
async def scrape():

    # Iniciamos Playwright
    async with async_playwright() as pw:

        # Lanzamos el navegador Chromium en modo headless (sin interfaz gráfica)
        browser = await pw.chromium.launch(headless=True)

        # Abrimos una nueva pestaña/página del navegador
        page = await browser.new_page()

        # Navegamos a la página objetivo
        await page.goto('https://books.toscrape.com')

        # Esperamos a que aparezcan los elementos de los libros en la página
        await page.wait_for_selector('article.product_pod')

        # Seleccionamos todos los elementos de libros en la página
        libros = await page.query_selector_all('article.product_pod')

        # Recorremos cada libro encontrado
        for libro in libros:

            # Buscamos el enlace del título dentro del libro
            titulo_element = await libro.query_selector('h3 a')

            # Extraemos el atributo "title" (el nombre del libro)
            titulo = await titulo_element.get_attribute('title')

            # Buscamos el elemento del precio
            precio_element = await libro.query_selector('.price_color')

            # Extraemos el texto del precio
            precio = await precio_element.inner_text()

            # Mostramos por pantalla el resultado
            print(f"{titulo} → {precio}")

        # Cerramos el navegador al terminar el scraping
        await browser.close()

# Ejecutamos la función asíncrona
await scrape()

Acabamos de observar como sólamente nos ha cargado la primera página. Hemos extraído todos los libros visibles en ese momento.
Sin embargo, en muchos sitios web la información no está toda en una sola página, sino que se divide en varias páginas mediante paginación.

Para resolver esto, necesitamos dar un paso más y automatizar la navegación entre páginas, como haría un usuario real al hacer clic en “Next”.

Esto nos permite:

📄 Recorrer todas las páginas del catálogo


📦 Extraer todos los datos disponibles en el sitio


🤖 Automatizar completamente la navegación

In [ ]:
from playwright.async_api import async_playwright

async def scrape():

    async with async_playwright() as pw:

        # Lanzamos el navegador Chromium en modo headless
        browser = await pw.chromium.launch(headless=True)
        page = await browser.new_page()

        # Accedemos a la página inicial
        await page.goto("https://books.toscrape.com")

        # Recorremos todas las páginas mediante paginación
        while True:

            # Esperamos a que carguen los libros en la página
            await page.wait_for_selector("article.product_pod")

            # Seleccionamos todos los libros visibles
            libros = await page.query_selector_all("article.product_pod")

            # Extraemos título y precio de cada libro
            for libro in libros:

                titulo = await libro.query_selector("h3 a")
                titulo = await titulo.get_attribute("title")

                precio = await libro.query_selector(".price_color")
                precio = await precio.inner_text()

                print(f"{titulo} → {precio}")

            # Comprobamos si existe botón de "siguiente página"
            next_button = await page.query_selector("li.next a")

            # Si no hay más páginas, terminamos el bucle
            if not next_button:
                break

            # Clic para ir a la siguiente página
            await next_button.click()

            # Esperamos a que cargue completamente la nueva página
            await page.wait_for_load_state("networkidle")

# Ejecutamos la función asíncrona
await scrape()

# 🧰 ACCESO A DATOS CON API (JSON)

---

## 📦 API (JSON)
- Permite acceder a **datos estructurados directamente desde servidores**
- No es necesario descargar ficheros manualmente
- Muy usada para datos oficiales (INE, bancos, redes sociales, etc.)

> "`requests` realiza la petición, `r.json()` convierte la respuesta a Python"

---

#### 🌐 Petición a la API

- Se realiza una solicitud HTTP a un **endpoint**
- El servidor devuelve datos en formato JSON
- Los datos vienen ya estructurados
- Datos: Evolución de la población:

  https://servicios.ine.es/wstempus/js/ES/DATOS_SERIE/CP335?nult=10

---

#### 🛡️ Validación de la respuesta

- `raise_for_status()` comprueba si la petición ha funcionado
- Si hay error (404, 500...), el programa se detiene automáticamente
- Evita trabajar con datos incorrectos

---

#### 📦 Conversión a JSON

- La respuesta no es texto plano
- Se convierte a un objeto de Python con `r.json()`
- El resultado suele ser un **diccionario o lista**

---

#### 📊 Estructura de los datos

- Los datos suelen venir **anidados**
- Cada clave contiene información específica
- En el caso del INE, los valores están en `"Data"`

---

#### 🔎 Acceso a la información

```python
print(data["Data"][0])

In [ ]:
import requests
r = requests.get("https://servicios.ine.es/wstempus/js/ES/DATOS_SERIE/CP335?nult=10", timeout=10)
r.raise_for_status()
data = r.json()
print(data)
# 2. Inspeccionar estructura
print(data["Data"][0])

# 🧭 CRAWLER MULTIPÁGINA

---

### 🕷️ ¿Qué hace este ejercicio?

Este código implementa un **crawler web multipágina**, es decir, un programa que recorre automáticamente varias páginas de un sitio web y extrae información de cada una.

El objetivo es recorrer todo el catálogo de libros y recopilar datos como:
- 📖 Título
- 💰 Precio
- 📦 Disponibilidad

---

## 📚 Librerías utilizadas

#### 📦 `requests`
- Permite hacer peticiones HTTP a una web.
- Se usa para **descargar el HTML** de cada página.

---

#### 🍲 `BeautifulSoup` (bs4)
- Permite analizar el HTML descargado.
- Facilita la extracción de información específica (títulos, precios, enlaces…).

---

#### 🧭 `deque` (collections)
- Estructura tipo **cola (queue)**.
- Permite recorrer páginas de forma ordenada (tipo BFS).
- Ideal para crawlers multipágina.

---

### 🌐 Página objetivo

Estamos recorriendo:

👉 https://books.toscrape.com/catalogue/page-1.html

La web está organizada en múltiples páginas conectadas mediante un botón de **“Next”**.

---

## 🔁 ¿Cómo funciona el crawler?

El flujo del programa es el siguiente:

1. Empieza en la primera página del catálogo.
2. Descarga el HTML con `requests`.
3. Extrae los libros de la página.
4. Guarda la información en una lista.
5. Busca el enlace a la siguiente página.
6. Si existe, lo añade a la cola.
7. Repite el proceso hasta terminar todas las páginas.

---

In [ ]:
import requests  # Para hacer peticiones HTTP a la web
from bs4 import BeautifulSoup  # Para analizar el HTML y extraer datos
from collections import deque  # Para usar una cola (FIFO) y recorrer páginas

# URL base del sitio web
BASE = 'https://books.toscrape.com'

# Cola de páginas por visitar (empezamos por la primera página del catálogo)
queue = deque([BASE + '/catalogue/page-1.html'])

# Conjunto para guardar URLs ya visitadas (evita repetir páginas)
visitados = set()

# Lista donde guardaremos los datos extraídos de los libros
resultados = []

# Mientras haya páginas en la cola...
while queue:
    # Sacamos la primera URL de la cola
    url = queue.popleft()

    # Si ya la hemos visitado, la ignoramos
    if url in visitados:
        continue

    # Marcamos la URL como visitada
    visitados.add(url)

    # Hacemos la petición HTTP y parseamos el HTML con BeautifulSoup
    soup = BeautifulSoup(requests.get(url).text, 'html.parser')

    # 🔎 EXTRAER LIBROS DE LA PÁGINA ACTUAL
    for libro in soup.select('article.product_pod'):
        resultados.append({
            # Título del libro (atributo title del enlace)
            'titulo': libro.select_one('h3 a')['title'],

            # Precio del libro (texto del elemento con clase price_color)
            'precio': libro.select_one('.price_color').get_text(strip=True),

            # Disponibilidad (texto del bloque availability)
            'disponible': libro.select_one('.availability').get_text(strip=True),
        })

    # 🔗 BUSCAR ENLACE A LA SIGUIENTE PÁGINA
    next_pg = soup.select_one('li.next a')

    # Si existe una página siguiente...
    if next_pg:
        # Construimos la URL completa y la añadimos a la cola
        queue.append(BASE + '/catalogue/' + next_pg['href'])

# 📊 RESULTADOS FINALES

# Mostramos el número total de libros encontrados
print(f"Total libros: {len(resultados)}")

# Mostramos solo los 5 primeros libros para comprobar
for r in resultados[:5]:
    print(r)

# 🚫 BLOQUEO DE IP
---


#### 📡 ¿Qué vamos a hacer?

Este código está haciendo:

-  100 peticiones seguidas a la API de GitHub
-  Sin pausas entre peticiones
- Desde la misma IP (tu conexión)
---
#### 📊 Posibles respuestas HTTP

200 → ✔️ Petición correcta  
403 → 🚫 Acceso denegado (posible bloqueo)  
429 → ⏳ Too Many Requests (rate limit alcanzado)



---

#### 🚦 ¿Qué hace GitHub?

GitHub (como la mayoría de APIs públicas):

-  Limita el número de peticiones por minuto/hora
-  Detecta comportamiento automatizado
-  Protege sus servidores de abusos

---



In [ ]:
import requests

url = "https://api.github.com/users/octocat"

for i in range(100):
    r = requests.get(url)
    print(i, r.status_code)


#### 🧠 ¿Qué está ocurriendo realmente?

- Algunas peticiones pueden devolver 200
- Después la API detecta exceso de tráfico
- Empieza a responder con 403 o 429

---

#### 🚫 ¿Qué es el “bloqueo de IP”?

Es una medida de seguridad donde:

-  Se identifica tu dirección IP
-  Se limita o rechaza tu tráfico
-  Se impide seguir haciendo peticiones durante un tiempo

---

#### 🧠 Conclusión

👉 Las APIs públicas no están diseñadas para scraping masivo sin control  
👉 El comportamiento humano (pausas) es clave  
👉 El exceso de requests puede activar bloqueos automáticos

# 🌐 SOLUCIONES: ROTACIÓN DE PROXYS Y RATE LIMITING

---

#### 🧠 ¿Qué vamos a hacer?

En este ejercicio vamos a aprender cómo funciona el uso de **proxies en web scraping**, una técnica utilizada para:

-  Simular diferentes direcciones IP
-  Evitar bloqueos básicos por demasiadas peticiones
-  Manejar fallos de conexión
-  Entender cómo funcionan sistemas distribuidos de scraping
---

#### 🌐 ¿Qué es un proxy?

Un proxy es un servidor intermedio que:

-  Oculta tu IP real
-  Hace la petición desde otra ubicación
-  Puede ayudar a evitar bloqueos simples

---
#### ⏳ ¿Qué es el Rate Limiting?

El **rate limiting** es un mecanismo de seguridad que usan las APIs y webs para controlar cuántas peticiones puede hacer un usuario en un tiempo determinado.

---

#### 🚦 Objetivo del rate limiting:

-  Evitar abusos o saturación del servidor  
-  Proteger la infraestructura  
-  Garantizar acceso justo a todos los usuarios  

---

In [ ]:
import requests
import time
import random

PROXIES = [ "http://proxy_malo_1:8080",
           "http://proxy_malo_2:8080",
            None # este es el bueno (sin proxy o proxy funcional real)
]

def fetch(url):

    for i, proxy in enumerate(PROXIES):

        try:
            print(f"\n🔁 Intento {i+1} usando proxy: {proxy}")

            r = requests.get(
                url,
                proxies={"http": proxy, "https": proxy} if proxy else None,
                timeout=3
            )

            r.raise_for_status()

            # ⏳ RATE LIMITING (control de ritmo)
            time.sleep(1 + random.uniform(0, 0.5))

            print("✅ ÉXITO")
            return r.text

        except Exception as e:
            print(f"❌ Falló proxy {proxy}: {e}")
            time.sleep(1)  # pausa entre intentos

    print("\n💥 Todos los proxies fallaron")
    return None
html = fetch("https://books.toscrape.com")

if html:
    print("\n📄 HTML cargado correctamente")

# 🚦 CONTROL DE LÍMITE DE PETICIONES (HTTP 429) EN WEB SCRAPING

---

#### 🧠 ¿Qué vamos a aprender?

En este ejercicio vamos a simular cómo gestionar el **límite de peticiones (rate limiting)** en web scraping.

Cuando hacemos demasiadas peticiones a una web, el servidor puede bloquearnos temporalmente.

---

#### 🚫 ¿Qué es un error 429?

El código HTTP:

> 🔴 **429 Too Many Requests**

significa que:

- 🚧 Has hecho demasiadas peticiones en poco tiempo
- ⛔ El servidor te está limitando temporalmente
- ⏳ Debes esperar antes de volver a intentar

---

#### 🧰 Técnica usada: Exponential Backoff

Para evitar bloqueos usamos:

> 📈 **Backoff exponencial**

Esto significa:

- Cada intento espera más tiempo que el anterior
- Se añade un pequeño aleatorio para evitar patrones
---

In [ ]:
url = "https://api.github.com/users/octocat"

for i in range(500):
    r = requests.get(url)

In [ ]:
def get_with_backoff(max_intentos=4):

    for intento in range(max_intentos):

        r = requests.get(url)

        print(f"Intento {intento+1} → {r.status_code}")

        if r.status_code in (429, 403):

            espera = (2 ** intento) + random.uniform(0, 1)

            print(f"⏳ Bloqueado. Esperando {espera:.2f}s...\n")

            time.sleep(espera)
            continue

        return "OK"

    raise Exception("❌ BLOQUEO PERMANENTE (rate limit alcanzado)")


get_with_backoff()

# 🚧 WEB SCRAPING Y BLOQUEOS (CAPTCHA / ANTI-BOT)

---

#### 🧠 ¿Qué vamos a ver en este ejercicio?

En este ejemplo vamos a intentar hacer **web scraping en una página con protección anti-bots**:

👉 https://www.scrapingcourse.com/antibot-challenge

Esta web suele tener:
- 🚫 sistemas anti-scraping
- 🤖 detección de bots
- 🧩 CAPTCHA o bloqueos
- 🔐 restricciones por tráfico automatizado

---


In [ ]:
import requests

url = "https://www.scrapingcourse.com/antibot-challenge"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
}

r = requests.get(url, headers=headers)

html = r.text.lower()

print("STATUS:", r.status_code)

# 🔍 detección básica de bloqueo
block_signals = [
    "just a moment",
    "checking your browser",
    "cf-challenge",
    "cloudflare",
    "verify you are human",
    "attention required"
]

print("\n--- DETECCIÓN ---")

if any(signal in html for signal in block_signals):
    print("🚨 BLOQUEO ANTI-BOT DETECTADO")
else:
    print("✅ NO parece bloqueo (contenido normal)")
html

# 🔐 AUTENTICACIÓN, SESIONES Y COOKIES EXPIRADAS EN WEB SCRAPING

---

#### 🧠 ¿Qué vamos a ver en este ejercicio?

En este ejemplo vamos a hacer una petición HTTP a una web real:

👉 https://www.linkedin.com/in/

El objetivo es entender qué ocurre cuando intentamos acceder a una página que:

- 🔐 requiere autenticación (login)
- 🍪 depende de cookies de sesión válidas
- 🚫 bloquea acceso sin credenciales
- ⏳ puede devolver contenido diferente según el estado del usuario

---

In [ ]:
import requests

r = requests.get("https://www.linkedin.com/in/")

print(r.status_code)
print(r.text[:1000])

# ⚖️ WEB SCRAPING Y ROBOTS.TXT (RESTRICCIONES)

---

#### 🧠 ¿Qué vamos a comprobar?

Vamos a ver si una web permite scraping revisando su archivo:

> 🤖 robots.txt

---

## 🌐 Enlaces reales

#### 📚 Books to Scrape (práctica)
👉 https://books.toscrape.com/robots.txt  
✔️ Web de aprendizaje, no tiene robots.txt

---

#### INE (Instituto Nacional de Estadística)
👉 https://www.ine.es/robots.txt  
✔️ Generalmente permite acceso a muchos recursos públicos  
⚠️ Aun así puede haber límites por carga o uso abusivo

---

#### 🔐 LinkedIn (restringido)
👉 https://www.linkedin.com/robots.txt  
🚫 Bloquea el acceso a muchas rutas como perfiles y búsquedas

---

#### 🚫 ¿Por qué NO se puede scrapear LinkedIn?

- 🔐 Protege datos personales (perfiles profesionales)
- ⚖️ Lo prohíben sus términos de uso (ToS)
- 🤖 Bloquea bots en robots.txt
- 🚧 Usa medidas anti-scraping (detección, CAPTCHAs, IP blocks)

---

#### 🧠 IDEA CLAVE

> 🤖 robots.txt indica si se permite rastreo automático, pero no sustituye la ley ni los términos de uso

In [ ]:
from urllib.robotparser import RobotFileParser  # Permite leer y analizar robots.txt

# 🔎 Función para comprobar si se puede scrapear una URL según robots.txt
def puedo_scrapear(url_base, url_objetivo, user_agent='*'):

    # 🤖 Creamos el parser de robots.txt
    rp = RobotFileParser()

    # 🌐 Indicamos la ruta del robots.txt de la web
    rp.set_url(url_base + '/robots.txt')

    # 📥 Descargamos y leemos el archivo robots.txt
    rp.read()

    # 🧭 Comprobamos si el user-agent puede acceder a la URL objetivo
    permitido = rp.can_fetch(user_agent, url_objetivo)

    # 📊 Mostramos resultado en pantalla
    print(f"{'✅ Permitido' if permitido else '❌ Prohibido'}: {url_objetivo}")

    # 🔙 Devolvemos True/False
    return permitido


# 🌐 📚 Books to Scrape (web de práctica)
puedo_scrapear(
    'https://books.toscrape.com',
    'https://books.toscrape.com/catalogue/'
)

# 🌐  INE (Instituto Nacional de Estadística)
puedo_scrapear(
    'https://www.ine.es',
    'https://www.ine.es/jaxiT3/Tabla.htm'  # ejemplo de recurso de tablas estadísticas
)

# 🌐 🔐 LinkedIn (web con restricciones fuertes)
puedo_scrapear(
    'https://www.linkedin.com',
    'https://www.linkedin.com/in/'
)